# Sandro - Prostate158 - PROSTATEx external test of the final model

**Inference only** (no training). Runtime > Change runtime type > **T4 GPU**, then **Runtime > Run all**,
and allow Google Drive when asked (twice: once to mount Drive, once to read the shared bundle).

* Model: 2d softmax ensemble of seeds 42, 43, 44
  with left-right flip TTA, frozen on the validation split.
* Cohort: PROSTATEx, 204 cases, downloaded from the public source at a pinned commit, every file checked by SHA-256.
* Expected: about 30-60 min on a T4, about 1-2 compute units.
* Results go to `MyDrive/Prostate158_E1_seeds/final_prostatex/` and `final_prostatex.zip` next to it.
  If Colab disconnects, Run all again: finished cases are kept on Drive and skipped.

In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/Prostate158_E1_seeds/final_prostatex'


In [ ]:
# Download the bundle shared with you (code + 3 frozen checkpoints + manifest).
import hashlib, io, os, zipfile
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
BUNDLE_NAME, BUNDLE_SHA256 = 'prostatex_final_bundle.zip', 'b94204e7b0a07646216eb68096df15c031b0604821279648d4689213feee559a'
BUNDLE = '/content/' + BUNDLE_NAME
def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()
if not (os.path.exists(BUNDLE) and sha256(BUNDLE) == BUNDLE_SHA256):
    auth.authenticate_user()
    svc = build('drive', 'v3')
    hits = svc.files().list(q=f"name = '{BUNDLE_NAME}' and trashed = false",
                            fields='files(id, name, size)', includeItemsFromAllDrives=True,
                            supportsAllDrives=True).execute()['files']
    assert hits, 'Bundle not found: open the shared folder link once, then run again.'
    req = svc.files().get_media(fileId=hits[0]['id'])
    with io.FileIO(BUNDLE, 'wb') as fh:
        dl = MediaIoBaseDownload(fh, req, chunksize=64 << 20)
        done = False
        while not done:
            status, done = dl.next_chunk()
            print(f'  {int(status.progress() * 100)}%', end='')
    print()
assert sha256(BUNDLE) == BUNDLE_SHA256, 'Bundle checksum mismatch'
WORK = '/content/bundle'
zipfile.ZipFile(BUNDLE).extractall(WORK)
os.chdir(WORK)
print('bundle OK:', BUNDLE_SHA256[:16])


In [ ]:
# PROSTATEx images + masks from the public source at the pinned commit; every file is checked.
import csv, json, urllib.request
from concurrent.futures import ThreadPoolExecutor
ROOT = 'dataset/external/prostatex'
prov = json.load(open(f'{ROOT}/metadata/acquisition_provenance.json'))
BASE = f"https://raw.githubusercontent.com/rcuocolo/PROSTATEx_masks/{prov['git_commit']}/Files/prostate"
SRC = {'raw/images/t2': 'Images', 'raw/masks/pz': 'mask_pz', 'raw/masks/rest': 'mask_tz'}
expected = {}
for line in open(f'{ROOT}/metadata/checksums.sha256'):
    digest, rel = line.split()
    expected[rel] = digest
def fetch(rel):
    path = os.path.join(ROOT, rel)
    if os.path.exists(path) and sha256(path) == expected[rel]:
        return rel, True
    folder, name = os.path.dirname(rel), os.path.basename(rel)
    os.makedirs(os.path.join(ROOT, folder), exist_ok=True)
    url = f"{BASE}/{SRC[folder]}/{name}"
    for _ in range(4):
        try:
            urllib.request.urlretrieve(url, path)
            if sha256(path) == expected[rel]:
                return rel, True
        except Exception as e:
            print('retry', name, e)
    return rel, False
with ThreadPoolExecutor(16) as ex:
    res = list(ex.map(fetch, sorted(expected)))
bad = [r for r, ok in res if not ok]
assert not bad, f'{len(bad)} files failed the checksum: {bad[:5]}'
print(f'{len(res)} PROSTATEx files downloaded and verified')


In [ ]:
# Inference with the frozen ensemble (member hashes are checked against the selection record).
import sys
cmd = [sys.executable, '-u', 'scripts/run_prostatex_external_validation.py',
       '--prostatex-root', ROOT, '--checkpoint', 'results/e1_seed_selection/selection.json',
       '--output-dir', OUT_DIR, '--work-dir', '/content/prostatex_harmonized', '--device', 'cuda']
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='')
assert p.wait() == 0, 'run failed: Run all again to resume'


In [ ]:
# Pack the results for Sandro (reconstructions included, harmonized copies of the data excluded).
import shutil
zip_path = shutil.make_archive(OUT_DIR, 'zip', OUT_DIR)
print('Done. Send this back:', zip_path, f'({os.path.getsize(zip_path) / 1e6:.0f} MB)')
s = json.load(open(os.path.join(OUT_DIR, 'summary.json')))
for c in ('1', '2'):
    d = s['summary'][c]['metrics']['dice']
    print(s['summary'][c]['class_name'], 'Dice', round(d['mean'], 4), 'n', d['n_cases'])
